In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
%cd /content/drive/MyDrive/ML_Portfolio/retail-sales-forecasting

/content/drive/MyDrive/ML_Portfolio/retail-sales-forecasting


In [9]:
import os
os.getcwd()

'/content/drive/MyDrive/ML_Portfolio/retail-sales-forecasting'

# Phase 6 — Machine Learning Modeling and Forecasting

This phase focuses on building and evaluating machine learning models for retail sales forecasting.

The objective is to predict future weekly sales using the engineered forecasting features created during the previous phase.

The modeling phase includes:

* linear models
* tree-based ensemble models
* Prophet forecasting

Different preprocessing approaches will also be compared to evaluate their impact on forecasting performance, including:

* scaled vs unscaled features
* original vs transformed target variables

The models will be evaluated using chronological forecasting data to simulate real-world future prediction scenarios while preventing data leakage.

Performance comparison will be based on forecasting evaluation metrics such as:

* MAE
* RMSE
* R² Score

The final goal is to identify the most effective forecasting approach for predicting retail sales behavior while also understanding the influence of preprocessing techniques and feature engineering on model performance.


In [2]:
%pip install prophet
%pip install joblib
%pip install xgboost
%pip install lightgbm
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


In [3]:
# data handling
import pandas as pd
import numpy as np

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# linear models
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

# tree-based models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

# boosting models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# model evaluation
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error
)

# hyperparameter tuning
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

#preprocessing
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline

# import prophet
from prophet import Prophet

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [6]:
import os
os.getcwd()

'/content'

In [11]:
# load processe data
X_train = pd.read_csv('data/processed/X_train_unscaled.csv')
X_test = pd.read_csv('data/processed/X_test_unscaled.csv')

X_train_scaled = pd.read_csv('data/processed/X_train_scaled.csv')
X_test_scaled = pd.read_csv('data/processed/X_test_scaled.csv')

y_train = pd.read_csv('data/processed/y_train_unscaled.csv')
y_test = pd.read_csv('data/processed/y_test_unscaled.csv')

y_train_log = pd.read_csv('data/processed/y_train_log.csv')
y_test_log = pd.read_csv('data/processed/y_test_log.csv')

In [12]:
# convert dataframe to series on target
y_train = y_train.squeeze()
y_test = y_test.squeeze()

y_train_log = y_train_log.squeeze()
y_test_log = y_test_log.squeeze()

In [13]:
# inspect dataset shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")
print(f"y_train_log shape: {y_train_log.shape}")
print(f"y_test_log shape: {y_test_log.shape}")

X_train shape: (268488, 27)
X_test shape: (127116, 27)
y_train shape: (268488,)
y_test shape: (127116,)
X_train_scaled shape: (268488, 27)
X_test_scaled shape: (127116, 27)
y_train_log shape: (268488,)
y_test_log shape: (127116,)


## Baseline Forecast Model

In this section, a baseline forecasting model will be created to establish a minimum performance benchmark for the forecasting task.

A naive forecasting approach will be used where the previous week's sales (`Lag_1`) are used to predict the current week's sales.

This baseline model will serve as a reference point for evaluating whether more advanced machine learning models are able to improve forecasting performance.


In [14]:
# create baseline predictions using previous week's sales
baseline_pred = X_test['Lag_1']

#  calculate baseline evaluation metrics
baseline_mae = mean_absolute_error(y_test, baseline_pred)

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

baseline_r2 = r2_score(y_test, baseline_pred)

# display baseline results
print(f'Baseline MAE: {baseline_mae:.2f}')
print(f'Baseline RMSE: {baseline_rmse:.2f}')
print(f'Baseline R²: {baseline_r2:.4f}')

Baseline MAE: 1720.87
Baseline RMSE: 3945.25
Baseline R²: 0.9682


## Baseline Model Results

The baseline forecasting model produced strong performance using the previous week's sales (`Lag_1`) as the prediction for the current week's sales.

### Baseline Performance

* MAE: 1720.87
* RMSE: 3945.25
* R² Score: 0.9682

The high R² score indicates that weekly sales show strong temporal dependency, meaning recent sales behavior is highly predictive of future sales.

The relatively low forecasting errors suggest that sales patterns remain fairly consistent across consecutive weeks, although larger errors still occur during periods of higher sales volatility such as holidays and promotional periods.

These results establish a strong forecasting benchmark for evaluating the performance of the advanced machine learning models.


## Automated Modeling Framework

Before training the machine learning models, an automated modeling framework will be developed to streamline the experimentation process.

The framework will be designed to:

* train multiple models automatically
* generate predictions
* evaluate forecasting performance
* store model evaluation metrics
* support preprocessing comparisons

This approach will improve workflow consistency, reduce repetitive code, and allow fair comparison across different forecasting models and preprocessing strategies.


In [16]:
import joblib

# load fitted transformer
power_transformer = joblib.load('model/power_transformer.pkl')

In [20]:
# store combine model result
model_results = []

# function to train and evaluate forecasting models
def run_models(
    models,
    X_train,
    y_train,
    X_test,
    y_test,
    feature_type,
    target_type,
    inverse_transformer=None
):
    """
    Train multiple forecasting models,
    generate predictions,
    evaluate performance,
    and return model results.
    """

    # store experiment results
    results = []

    # loop through models
    for model_name, model in models.items():

        # train model
        model.fit(X_train, y_train)

        # generate predictions
        predictions = model.predict(X_test)

        # inverse transform predictions if needed
        if inverse_transformer is not None:

            predictions = inverse_transformer.inverse_transform( predictions.reshape(-1, 1)).flatten()

            y_actual = inverse_transformer.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        else:
            y_actual = y_test

        # calculate evaluation metrics
        mae = mean_absolute_error(y_actual, predictions)

        rmse = np.sqrt(mean_squared_error(y_actual, predictions))

        r2 = r2_score(y_actual, predictions)

        # store results
        results.append({

            'Model': model_name,
            'Feature_Type': feature_type,
            'Target_Type': target_type,
            'MAE': mae,
            'RMSE': rmse,
            'R2_Score': r2
        })

        # display progress
        print(f'{model_name} completed')

    return results

## Linear Model Experiments

In this section, multiple linear regression models will be trained and evaluated using different preprocessing strategies.

The experiments will compare:

* unscaled features with original target values
* scaled features with original target values
* scaled features with transformed target values

The objective is to examine how feature scaling and target transformation influence forecasting performance in linear models.

The following models will be evaluated:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Elastic Net Regression
## Linear Model Experiments

In this section, multiple linear regression models will be trained and evaluated using different preprocessing strategies.

The experiments will compare:

* unscaled features with original target values
* scaled features with original target values
* scaled features with transformed target values

The objective is to examine how feature scaling and target transformation influence forecasting performance in linear models.

The following models will be evaluated:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Elastic Net Regression


In [18]:
# linear regression models
linear_models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression' : Ridge(),
    'Lasso Regression' : Lasso(),
    'ElasticNet' : ElasticNet()
}

## **Experiment 1: Unscaled Features + Original Target**

In [19]:
# run linear models on unscaled features
linear_unscaled_result = run_models(
    models=linear_models,

    X_train=X_train,
    X_test=X_test,

    y_train=y_train,
    y_test=y_test,

    feature_type="Unscaled",
    target_type='Original'
)

Linear Regression completed
Ridge Regression completed
Lasso Regression completed
ElasticNet completed


## **Experiment 2: Scaled Features + Original Target**

In [21]:
# run linear models on scaled features
linear_scaled_result = run_models(
    models=linear_models,

    X_train=X_train_scaled,
    X_test=X_test_scaled,

    y_train=y_train,
    y_test=y_test,

    feature_type="Scaled",
    target_type="Original"
)

Linear Regression completed
Ridge Regression completed
Lasso Regression completed
ElasticNet completed


## **Experiment 3: Scaled Features + Transformed Target**

In [22]:
# run linear models on transformed target
linear_transformed_result = run_models(
    models=linear_models,

    X_train=X_train_scaled,
    X_test=X_test_scaled,

    y_train=y_train_log,
    y_test=y_test_log,

    feature_type="Scaled",
    target_type="Transformed",

    inverse_transformer=power_transformer
)

Linear Regression completed
Ridge Regression completed
Lasso Regression completed
ElasticNet completed


In [23]:
# combine linear model results
linear_results = (linear_unscaled_result + linear_scaled_result + linear_transformed_result)

In [24]:
# create linear model results dataframe
linear_results_df = pd.DataFrame(linear_results)

# display results
linear_results_df.sort_values(by='RMSE')

,Model,Feature_Type,Target_Type,MAE,RMSE,R2_Score
3,ElasticNet,Unscaled,Original,1971.115200,3982.030956,0.967635
6,Lasso Regression,Scaled,Original,2281.008215,4136.230868,0.965080
2,Lasso Regression,Unscaled,Original,2282.047937,4136.875723,0.965069
1,Ridge Regression,Unscaled,Original,2314.378960,4154.569101,0.964770
5,Ridge Regression,Scaled,Original,2314.403846,4154.572974,0.964770
0,Linear Regression,Unscaled,Original,2314.522998,4154.650207,0.964768
4,Linear Regression,Scaled,Original,2314.522998,4154.650207,0.964768
7,ElasticNet,Scaled,Original,2791.560122,4780.237778,0.953359
9,Ridge Regression,Scaled,Transformed,3582.004085,7290.605276,0.891510
8,Linear Regression,Scaled,Transformed,3582.022958,7290.671736,0.891508


## Linear Model Results and Observations

The linear models performed strongly on the original target variable across both scaled and unscaled feature configurations.

Feature scaling had little to no impact on model performance, as the evaluation metrics remained almost identical after scaling. This suggests that the engineered forecasting features were already within reasonable numerical ranges for the linear models.

The models trained on the original target variable consistently outperformed the transformed target experiments. Applying the Yeo-Johnson transformation reduced forecasting performance across all linear models, indicating that transforming the sales values may have weakened important temporal sales patterns contained in the original data.

Among the linear models, Elastic Net achieved the lowest RMSE on the original target experiment with an RMSE of 3982.03 and an R² score of 0.9676. Ridge Regression and standard Linear Regression also produced stable forecasting performance.

However, the naive baseline forecasting model still achieved the strongest overall performance with:

* RMSE: 3945.25
* MAE: 1720.87
* R² Score: 0.9682

This indicates that previous sales values contain extremely strong predictive information within the dataset, making the forecasting task highly dependent on recent historical sales behavior.


## Tree-Based Model Experiments

In this section, tree-based regression models will be trained and evaluated using the engineered forecasting features.

Unlike linear models, tree-based models are generally less sensitive to feature scaling because they split data based on feature thresholds rather than distance calculations.

The experiments will compare:

* unscaled features with original target values
* unscaled features with transformed target values

The objective is to examine whether tree-based models are able to capture nonlinear forecasting patterns more effectively than the linear regression models.

The following models will be evaluated:

* Decision Tree Regressor
* Random Forest Regressor


In [25]:
# tree-based regression models
tree_models = {
    'Random Forest' : RandomForestRegressor(random_state=42),
    'Decision Tree' : DecisionTreeRegressor(random_state=42)
}

## **Experiment 1: Original Target Experiment**

In [26]:
# run tree models on original target
tree_original_result = run_models(
    models=tree_models,

    X_train=X_train,
    X_test=X_test,

    y_train=y_train,
    y_test=y_test,

    feature_type='Unscaled',
    target_type="Original"
)

Random Forest completed
Decision Tree completed


## **Experiment 2: Transformed Target Experiment**

In [ ]:
# run tree models on transformed target
tree_transformed_results = run_models(
    models=tree_models,

    X_train=X_train,
    X_test=X_test,

    y_train=y_train_log,
    y_test=y_test_log,

    feature_type="Unscaled",
    target_type="Transformed",

    inverse_transformer=power_transformer

)

Random Forest completed
Decision Tree completed


In [ ]:
# combine tree model results
tree_results = (tree_original_result + tree_transformed_results)

# convert to dataframe
tree_results_df = pd.DataFrame(tree_results)

# # sort results by RMSE

tree_results_df.sort_values(by='RMSE')

,Model,Feature_Type,Target_Type,MAE,RMSE,R2_Score
2,Random Forest,Unscaled,Transformed,1530.035392,3256.624243,0.978353
0,Random Forest,Unscaled,Original,1569.363211,3329.240115,0.977377
3,Decision Tree,Unscaled,Transformed,2423.076844,5320.777203,0.942215
1,Decision Tree,Unscaled,Original,2525.278480,5927.824018,0.928278
